# Sentinel-2 MSI L2A Download & Extraction

This notebook demonstrates how to search for Sentinel-2 L2A granules via **Planetary Computer STAC**, and extract RGB imagery using the **aer-extract-pc-sentinel2** plugin.

The output is written as GeoTIFFs in the EOIDS structure so they can be mosaicked with `mosaic_eoids_tiles`.

## Setup

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
import time

import geopandas as gpd

from aer.client import AerClient
from aer.interfaces import ExtractionProfile

## Configuration

Sentinel-2 L2A data on Planetary Computer is available for historical dates.
We use **2025-04-01 to 2025-04-30** (~1 month) so that multiple revisit cycles fill in the mosaic with fewer holes.

In [ ]:
# --- Configuration ---
DATE_START = datetime(2025, 4, 1, 0, 0, tzinfo=timezone.utc)
DATE_END = datetime(2025, 4, 30, 0, 0, tzinfo=timezone.utc)

# Load AOI (Buenos Aires province)
geojson_path = Path("buenos_aires.geojson")
gdf = gpd.read_file(geojson_path)
aoi = gdf.geometry.iloc[0]

# --- Client Setup ---
client = AerClient()

## Search

Search the Planetary Computer STAC catalog for `sentinel-2-l2a` assets.

In [ ]:
print("Searching...", flush=True)
results = client.search(
    collections=["sentinel-2-l2a"],
    start_datetime=DATE_START,
    end_datetime=DATE_END,
    intersects=aoi,
)
print(f"Found {len(results)} results", flush=True)
results.head()

## Prepare Extraction

Define the extraction profile and prepare tasks.

We extract bands **B04** (Red), **B03** (Green), and **B02** (Blue) at 100 m resolution.

In [ ]:
# --- Prepare Extraction ---
uri = "extract_buenos_aires_sentinel2"

profiles = [
    ExtractionProfile(
        name="s2_rgb",
        resolution=100,
        collection_variables_map={"sentinel-2-l2a": ["B04", "B03", "B02"]},
    )
]

tasks = client.prepare_for_extraction(
    results,
    target_aoi=aoi,
    uri=uri,
    profiles=profiles,
    target_grid_dist=256000,
    target_grid_overlap=False,
    prepare_params={"cells_per_chunk": 10},
)

print(f"Prepared {len(tasks)} extraction tasks", flush=True)

## Extract

Run the extraction.  Each task downloads the required Sentinel-2 COG assets and writes
GeoTIFFs into the EOIDS folder tree.

In [ ]:
# Clean output directory
uri_path = Path(uri)
if uri_path.exists():
    shutil.rmtree(uri_path)
uri_path.mkdir(parents=True)

print("Extracting...", flush=True)
start_time = time.time()

results_df = client.extract_batches(
    tasks,
    max_batch_workers=4,
)

end_time = time.time()
print(f"Extraction took {end_time - start_time:.2f}s")
print(f"Extracted {len(results_df)} artifacts")

## Results

In [ ]:
results_df[["id", "collection", "grid_cell", "uri"]].head()